### Imports

In [ ]:
from pathlib import Path

import pandas as pd

### Read the data from the Excel file

In [ ]:
RAW_PATH = Path("../data/raw")
assert RAW_PATH.exists(), "Create raw folder and add the necessary files inside."

In [ ]:
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(exist_ok=True)

In [ ]:
SIMULADOR_FILE = RAW_PATH / "Simulador Bloco e Subbloco_v2.xlsx"
assert SIMULADOR_FILE.exists(), "Missing Simulador file add to data/raw folder."

In [ ]:
excel = pd.ExcelFile(SIMULADOR_FILE, engine="calamine")
print(excel.sheet_names)

# Data cleaning steps

1. Extract 'Calendário FV' sheet from the provided Excel file.
    - extract sheet by sheet name
    - drop empty columns
    - drop index column
2. Extract 'Base Demanda' sheet from the provided Excel file.
    - extract sheet by sheet name
    - filter columns (not calculated columns)
    - merge with 'Calendário FV' sheet to get the 'Dia Captacao' column

### Extract calendar data from the 'Calendário FV' sheet

In [ ]:
df_calendario = excel.parse("Calendário FV")

In [ ]:
# drop empty columns
df_calendario = df_calendario.dropna(how="all", axis=1)

In [ ]:
df_calendario = df_calendario.drop(columns=["#"])

In [ ]:
df_calendario.columns

In [ ]:
# export csv
output_path = PROCESSED_PATH / "calendario_fv.csv"
df_calendario.to_csv(output_path, index=False)

### Extract demand from  'Base Demanda' sheet

Notes:
- Dia Captacao depends on data from calendar that seems to not be present in the 'Calendário FV' sheet. It is possible that this data is in another sheet or file. Further investigation is needed to locate the source of this information. Or even if this is needed. Initial thought it that it's not needed.

In [ ]:
df_demand = excel.parse("Base Demanda")

In [ ]:
DEMAND_COLUMNS_NEEDED = [
    "data_pedido",
    "cd_setor",
    "cd_cd",
    "nm_ciclo",
    "aa_ciclo",
    "total_pedidos",
    "total_volumes",
    "total_itens",
]
df_demand = df_demand[DEMAND_COLUMNS_NEEDED]

In [ ]:
df_demand["ciclo"] = df_demand["aa_ciclo"].astype(str) + df_demand["nm_ciclo"].astype(
    str
).str.zfill(2)

In [ ]:
df_calendario["CICLOS"] = df_calendario["CICLOS"].astype(str)

In [ ]:
n_before = len(df_demand)
df = df_demand.merge(
    df_calendario,
    left_on=["ciclo", "cd_setor"],
    right_on=["CICLOS", "COD SETOR"],
    how="left",
    indicator=True,
)
unmatched = df["_merge"].eq("left_only").sum()
print(f"{unmatched}/{n_before} demand rows unmatched to calendar ({unmatched / n_before:.1%})")

In [ ]:
df = df_demand.merge(df_calendario, left_on=["ciclo", "cd_setor"], right_on=["CICLOS", "COD SETOR"])

In [ ]:
output_file = PROCESSED_PATH / "demanda_with_calendar.csv"
df.to_csv(output_file, index=False)

### Extract the time series for the level forecasting

We need 1 point per (sector, cycle) combination with date = open date of the cycle. This will be used to forecast the level of demand for each sector and cycle. And, we need to aggregate the demand for each sector and cycle combination. The aggregation will be done by summing the demand for each sector and cycle combination. We need to include cycle duration in the time series. The cycle duration is the number of days between the open date and the close date of the cycle. The cycle duration will be used to forecast the level of demand for each sector and cycle.

In [ ]:
df["Dt Abertura"] = pd.to_datetime(df["Dt Abertura"])
df["Dt Fechamento"] = pd.to_datetime(df["Dt Fechamento"])

df_level = (
    df.groupby(["cd_setor", "ciclo"], as_index=False)
    .agg(
        date=("Dt Abertura", "first"),
        cycle_duration=("Qtde dias", "first"),
        total_pedidos=("total_pedidos", "sum"),
        total_volumes=("total_volumes", "sum"),
        total_itens=("total_itens", "sum"),
    )
    .sort_values(by=["cd_setor", "date"])
    .reset_index(drop=True)
)
df_level.head()

In [ ]:
output_file = PROCESSED_PATH / "demanda_level.csv"
df_level.to_csv(output_file, index=False)

### Extract the data for shape forecasting

We need one data point per (sector, cycle, position relative to the cycle start date normalized to the cycle duration) combination. This will be used to forecast the shape of the demand for each sector and cycle. The position relative to the cycle start date normalized to the cycle duration will be used to forecast the shape of the demand for each sector and cycle. Then, we need to show orders for each position as a share of the total orders for the cycle. This will be used to forecast the shape of the demand for each sector and cycle. The share of orders for each position will be used to forecast the shape of the demand for each sector and cycle. 

In terms of how many data points, we can use one per day with order, then we need to transform the date to position relative to the cycle start date normalized to the cycle duration. Then, we need to show orders for each position as a share of the total orders for the cycle.

So first, we need to extract the cycle totals, then we need to extract the position relative to the cycle start date normalized to the cycle duration, then we need to show orders for each position as a share of the total orders for the cycle. 

Since we don't really know whether items, orders or volume is the best measure of demand, we can extract all three and then decide which one to use for the shape forecasting.

In [ ]:
df_shape = df.copy()
# df_shape["cycle_total"] =
df_shape.columns

In [ ]:
df_shape["data_pedido"] = pd.to_datetime(df_shape["data_pedido"])
df_shape["Dt Abertura"] = pd.to_datetime(df_shape["Dt Abertura"])

In [ ]:
df_shape["relative_date"] = (df_shape["data_pedido"] - df_shape["Dt Abertura"]).dt.days / df[
    "Qtde dias"
]

In [ ]:
# exclude orders outside of cycle (don't know if that's the best way to deal with it)
df_shape = df_shape[df_shape["relative_date"] <= 1]

In [ ]:
# daily aggregation: collapses multiple distribution centers / records on the same day for a sector
# into single daily totals
df_shape = df_shape.groupby(
    ["cd_setor", "ciclo", "data_pedido", "relative_date"],
    as_index=False,
)[["total_pedidos", "total_volumes", "total_itens"]].sum()

# extract cycle totals (items, orders, volume)
df_shape[["ciclo_total_pedidos", "ciclo_total_volumes", "ciclo_total_itens"]] = df_shape.groupby(
    ["cd_setor", "ciclo"]
)[["total_pedidos", "total_volumes", "total_itens"]].transform("sum")

# share conversion: convert per day (items, orders, volume) to share of cycle total per sector
df_shape["share_pedidos"] = df_shape["total_pedidos"] / df_shape["ciclo_total_pedidos"]
df_shape["share_volumes"] = df_shape["total_volumes"] / df_shape["ciclo_total_volumes"]
df_shape["share_itens"] = df_shape["total_itens"] / df_shape["ciclo_total_itens"]

df_shape.head()

In [ ]:
output_file = PROCESSED_PATH / "demanda_shape.csv"
df_shape.to_csv(output_file, index=False)

# Exploration

In [ ]:
# I expect that every cycle start and end at the same day if the sector is in
# the same block + sublock
# this is the code to check this hypothesis is correct
is_consistent = (
    df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[["Dt Abertura", "Dt Fechamento"]]
    .nunique()
    .eq(1)
    .all()
    .all()
)

print(f"Expectation holds: {is_consistent}")

In [ ]:
# which blocks/subblocks/cycles diverge (and which sectors differ):
# Count unique start and end dates per block + subblock + cycle
cycle_dates_summary = df_calendario.groupby(["BLOCO", "SUB BLOCO", "CICLOS"])[
    ["Dt Abertura", "Dt Fechamento"]
].nunique()

# Filter for groups that have more than 1 distinct date
discrepancies = cycle_dates_summary[
    (cycle_dates_summary["Dt Abertura"] > 1) | (cycle_dates_summary["Dt Fechamento"] > 1)
]

if discrepancies.empty:
    print(
        "Confirmed: Every cycle starts and ends on the same day for "
        "all sectors in the same block + sublock."
    )
else:
    print(
        f"Found {len(discrepancies)} (BLOCO, SUB BLOCO, CICLOS) combination(s) "
        "with diverging dates:\n"
    )
    display(discrepancies)

    # Inspect the exact sectors and dates causing the mismatch
    conflicting_rows = df_calendario.merge(
        discrepancies.reset_index()[["BLOCO", "SUB BLOCO", "CICLOS"]],
        on=["BLOCO", "SUB BLOCO", "CICLOS"],
    )
    display(
        conflicting_rows[
            [
                "BLOCO",
                "SUB BLOCO",
                "CICLOS",
                "COD SETOR",
                "Dt Abertura",
                "Dt Fechamento",
                "Qtde dias",
            ]
        ].sort_values(
            by=[
                "CICLOS",
                "BLOCO",
                "SUB BLOCO",
                "Qtde dias",
                "COD SETOR",
            ]
        )
    )

In [ ]:
# Qtde dias seems to be cycle length including start and end day
(
    (df_calendario["Dt Fechamento"] - df_calendario["Dt Abertura"]).dt.days + 1
    != df_calendario["Qtde dias"]
).sum()

In [ ]:
# many orders are done outside of cycle prescribed length
# right now we're just dropping those outside of the cycle,
# is this the best way to handle this?

In [ ]:
# I assume that each sector can only be part of one block / sublock per cycle
dupe_check = df_calendario.groupby(["COD SETOR", "CICLOS"])["Dt Abertura"].nunique()
assert (dupe_check <= 1).all(), "Multiple distinct open dates per (setor, ciclo)"